# verbx — Dataset Augmentation Notebook

Demonstrates using `verbx.api` to augment an audio dataset with realistic reverb variations for ML training pipelines.

**Requirements:** `pip install verbx`

---

In [ ]:
from pathlib import Path

from verbx.api import analyze_file, generate_ir, read_audio, render_file, write_audio
from verbx.config import RenderConfig
from verbx.ir import IRGenConfig

## 1. Render a single file with algorithmic reverb

In [ ]:
# Single-file render — adjust paths to match your data
report = render_file(
    infile="dry.wav",
    outfile="wet_algo.wav",
    config=RenderConfig(
        engine="algo",
        rt60=2.5,
        pre_delay_ms=20,
        wet=0.65,
        dry=0.35,
        fdn_lines=16,
        fdn_matrix="hadamard",
        lowcut=80,
        highcut=16000,
    ),
)

print(f"Engine : {report['engine']}")
print(f"SR     : {report['sr']} Hz")
print(f"Samples: {report['input_samples']} → {report['output_samples']}")

## 2. Sweep RT60 values across a dataset directory

In [ ]:
DRY_DIR = Path("data/dry")       # edit to your dry source directory
WET_DIR = Path("data/augmented")
WET_DIR.mkdir(parents=True, exist_ok=True)

RT60_VALUES = [0.3, 0.8, 1.5, 3.0, 6.0]

for wav in sorted(DRY_DIR.glob("*.wav")):
    for rt60 in RT60_VALUES:
        tag = f"rt60_{rt60:.1f}".replace(".", "p")
        out = WET_DIR / f"{wav.stem}_{tag}.wav"
        render_file(
            infile=wav,
            outfile=out,
            config=RenderConfig(
                engine="algo",
                rt60=rt60,
                wet=0.7,
                dry=0.3,
                fdn_lines=16,
            ),
        )
        print(f"  {wav.name} → {out.name}")

## 3. Generate a synthetic IR and convolve

In [ ]:
# Generate a 4-second FDN impulse response at 48 kHz
ir_audio, ir_sr, ir_meta = generate_ir(
    IRGenConfig(
        mode="fdn",
        duration=4.0,
        sr=48000,
        channels=2,
        rt60=2.8,
    )
)

print(f"IR shape  : {ir_audio.shape}")
print(f"IR SR     : {ir_sr} Hz")
print(f"IR mode   : {ir_meta.get('mode')}")
print(f"IR RT60   : {ir_meta.get('rt60')} s")

# Save the IR for reuse
write_audio("synthetic_ir.wav", ir_audio, ir_sr)

# Convolve a dry file with the saved IR
render_file(
    infile="dry.wav",
    outfile="wet_conv.wav",
    config=RenderConfig(engine="conv", ir="synthetic_ir.wav", wet=0.8, dry=0.2),
)

## 4. Analyze and compare dry vs wet

In [ ]:
dry_metrics = analyze_file("dry.wav", include_loudness=True)
wet_metrics = analyze_file("wet_algo.wav", include_loudness=True)

keys = ["rms_dbfs", "peak_dbfs", "duration", "spectral_centroid"]
print(f"{'metric':<24} {'dry':>10} {'wet':>10}")
print("-" * 46)
for k in keys:
    print(f"{k:<24} {dry_metrics[k]:>10.3f} {wet_metrics[k]:>10.3f}")

## 5. Operate on raw arrays

Read and write audio as numpy arrays for custom DSP or dataset loaders.

In [ ]:
audio, sr = read_audio("wet_algo.wav")
print(f"Shape: {audio.shape}, dtype: {audio.dtype}, SR: {sr}")

# Example: mono downmix
mono = audio.mean(axis=1, keepdims=True)
write_audio("wet_mono.wav", mono, sr)
print(f"Mono shape: {mono.shape}")

## 6. Cache-aware IR generation for large sweeps

In [ ]:
from verbx.api import generate_ir

CACHE = Path(".verbx_ir_cache")

for rt60 in [0.5, 1.0, 2.0, 4.0, 8.0]:
    ir, sr, meta = generate_ir(
        IRGenConfig(mode="fdn", duration=max(rt60 + 1.0, 2.0), rt60=rt60, sr=48000),
        cache_dir=CACHE,
    )
    print(f"rt60={rt60:.1f}s  shape={ir.shape}  from_cache={meta.get('cache_hit', False)}")